In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
import scipy.sparse

# Set plot style
sns.set_style('whitegrid')

In [ ]:
BASE_PATH = '/content/drive/MyDrive/Projects/GitHub/DeepDiveRecommenders'


In [ ]:
"""
Glimse of data files

ratings.csv
```
userId,movieId,rating,timestamp
1,1,4.0,964982703
1,3,4.0,964981247
1,6,4.0,964982224
```

movies.csv
```
movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
```

tags.csv
```
userId,movieId,tag,timestamp
2,60756,funny,1445714994
2,60756,Highly quotable,1445714996
2,60756,will ferrell,1445714992
```
"""

## 1. Data Loading

In [ ]:
# Define file paths (assuming they are in a 'data/raw/' directory)
ratings_path = f'{BASE_PATH}/data/raw/ratings.csv'
movies_path = f'{BASE_PATH}/data/raw/movies.csv'
tags_path = f'{BASE_PATH}/data/raw/tags.csv'

# Load the datasets
print("Loading datasets...")
ratings_df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path)
tags_df = pd.read_csv(tags_path)
print(f"Loaded {len(ratings_df)} ratings, {len(movies_df)} movies, and {len(tags_df)} tags.")


## 2. Initial Merging & Preprocessing

In [ ]:
# Merge ratings with movies to get movie titles
df = pd.merge(ratings_df, movies_df, on='movieId', how='left')

# we will focus on the ratings and movie content.
# The tags will be merged later during feature engineering.
print("\nSample of merged ratings and movies data:")
print(df.head())

## 3. Time-Aware Train-Test Split

In [ ]:
# We will split the data for each user based on their rating timestamps.
print("\nPerforming time-aware train-test split for each user...")

def time_aware_user_split(df, test_size=0.2):
    """
    Performs a time-aware split for each user in a vectorized manner.

    Architecture:
    1. Sorts the entire dataframe by user and timestamp.
    2. Calculates a cumulative count (rank) for each user's ratings.
    3. Calculates the total number of ratings for each user.
    4. Uses these counts to identify the split point for each user.
    5. Splits the dataframe based on this vectorized condition.

    This avoids any explicit Python loops, ensuring high performance on large datasets.
    """
    # Sort values to ensure timestamps are in order for each user
    df_sorted = df.sort_values(by=['userId', 'timestamp'])

    # Create a ranked list of ratings for each user (1st rating, 2nd, etc.)
    df_sorted['user_rating_rank'] = df_sorted.groupby('userId').cumcount() + 1

    # Get the total number of ratings for each user
    user_rating_counts = df_sorted.groupby('userId').size()

    # Map the total counts back to the main dataframe
    df_sorted['user_total_ratings'] = df_sorted['userId'].map(user_rating_counts)

    # The split point: any rating ranked higher than this is in the test set
    # (1 - test_size) gives the proportion for the training set
    df_sorted['split_point'] = (df_sorted['user_total_ratings'] * (1 - test_size)).astype(int)

    # Boolean mask for the split
    # If a rating's rank is greater than the split point, it's in the test set
    is_test = df_sorted['user_rating_rank'] > df_sorted['split_point']

    # Split the data
    train_df = df_sorted[~is_test].drop(columns=['user_rating_rank', 'user_total_ratings', 'split_point'])
    test_df = df_sorted[is_test].drop(columns=['user_rating_rank', 'user_total_ratings', 'split_point'])

    # Drop users from test set who have no ratings in the training set
    # This can happen for users with very few ratings
    train_user_ids = train_df['userId'].unique()
    test_df = test_df[test_df['userId'].isin(train_user_ids)]

    return train_df, test_df


# Execute the split
train_df, test_df = time_aware_user_split(df)

print("\nSplit complete.")
print(f"Training set size: {len(train_df)} ratings")
print(f"Test set size: {len(test_df)} ratings")


## 4. Verification of the Split

In [ ]:
# Let's verify for a single user to ensure the logic is correct.
sample_user_id = 1
print(f"\nVerifying split for userId {sample_user_id}:")

# Get the original ratings for the user, sorted by time
original_user_ratings = df[df['userId'] == sample_user_id].sort_values('timestamp')
print("\nOriginal ratings (sorted by time):")
print(original_user_ratings[['userId', 'movieId', 'rating', 'timestamp']])

# Get the training ratings for the user
train_user_ratings = train_df[train_df['userId'] == sample_user_id]
print("\nTrain set ratings:")
print(train_user_ratings[['userId', 'movieId', 'rating', 'timestamp']])

# Get the test ratings for the user
test_user_ratings = test_df[test_df['userId'] == sample_user_id]
print("\nTest set ratings (should be the most recent):")
print(test_user_ratings[['userId', 'movieId', 'rating', 'timestamp']])

# A key check: the max timestamp in the train set should be less than the min in the test set
if not test_user_ratings.empty:
    max_train_timestamp = train_user_ratings['timestamp'].max()
    min_test_timestamp = test_user_ratings['timestamp'].min()
    assert max_train_timestamp < min_test_timestamp
    print("\nVerification successful: User's test ratings are indeed their most recent ones.")
else:
    print("\nUser has no ratings in the test set (as expected for users with few ratings).")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print("Starting Exploratory Data Analysis...")

# Plot the distribution of ratings
plt.figure(figsize=(10, 6))
sns.countplot(x='rating', data=train_df)
plt.title('Distribution of Movie Ratings in the Training Set')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

# Analyze the number of ratings per user
user_rating_counts = train_df.groupby('userId')['rating'].count()
plt.figure(figsize=(10, 6))
sns.histplot(user_rating_counts, bins=50, kde=True)
plt.title('Distribution of Number of Ratings per User')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Users')
plt.xlim(0, 500) # Zoom in on the majority of users
plt.show()

# Analyze the number of ratings per movie
movie_rating_counts = train_df.groupby('movieId')['rating'].count()
plt.figure(figsize=(10, 6))
sns.histplot(movie_rating_counts, bins=50, kde=True)
plt.title('Distribution of Number of Ratings per Movie')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Movies')
plt.xlim(0, 200) # Zoom in on the majority of movies
plt.show()

## 6. Genre Feature Engineering

In [ ]:
print("\nEngineering genre features...")

# First, handle movies with no listed genres
movies_df['genres'] = movies_df['genres'].fillna('No Genre Listed')

# Split the genres string into a list of genres
movies_df['genre_list'] = movies_df['genres'].apply(lambda x: x.split('|'))

# Use MultiLabelBinarizer to one-hot encode the genres
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(movies_df['genre_list'])

# Create a DataFrame with the genre features
genre_df = pd.DataFrame(genre_features, columns=mlb.classes_, index=movies_df['movieId'])

print("Genre features created. Shape:", genre_df.shape)
print("Sample of genre features:")
print(genre_df.head())

## 7. Tag Feature Engineering (TF-IDF)

In [ ]:
# --- 6. High-Performance Tag Feature Engineering (TF-IDF) ---
print("\nEngineering tag features (scalable method)...")

# Pre-process tags
tags_df['processed_tag'] = tags_df['tag'].str.lower().str.replace(' ', '', regex=False)

# Efficiently aggregate tags for each movie
# This avoids the slow .apply() method by using optimized string operations
movie_tags = tags_df.groupby('movieId')['processed_tag'].agg(list).str.join(' ').reset_index()
movie_tags.rename(columns={'processed_tag': 'tags_corpus'}, inplace=True)

# Merge the tags corpus back into the movies dataframe
movies_with_tags_df = pd.merge(movies_df, movie_tags, on='movieId', how='left')
movies_with_tags_df['tags_corpus'] = movies_with_tags_df['tags_corpus'].fillna('')

# Use TfidfVectorizer, which directly produces a sparse matrix
tfidf = TfidfVectorizer(max_features=5000)
tag_features_sparse = tfidf.fit_transform(movies_with_tags_df['tags_corpus'])

# Create a sparse-backed DataFrame for easy alignment without using excess memory
tag_sparse_df = pd.DataFrame.sparse.from_spmatrix(
    tag_features_sparse,
    index=movies_with_tags_df['movieId'],
    columns=tfidf.get_feature_names_out()
)

print("Tag features created (sparse matrix). Shape:", tag_features_sparse.shape)


## 8. Combine Features & Save Processed Data

In [ ]:
# 8.1 Create and Save the Final "Content Fingerprint"
print("\nCombining features and saving processed data (optimized formats)...")

# Align the genre and tag dataframes by movieId
aligned_genre_df, aligned_tag_df = genre_df.align(tag_sparse_df, join='left', axis=0, fill_value=0)

# Combine the features into a single sparse matrix
content_features_sparse = scipy.sparse.hstack([
    scipy.sparse.csr_matrix(aligned_genre_df.values),
    aligned_tag_df.sparse.to_coo() # Convert to COO format for hstack
])

# Ensure the combined matrix is in CSR format for efficient row slicing
content_features_sparse = content_features_sparse.tocsr()

print("Combined content features created (sparse matrix). Shape:", content_features_sparse.shape)

# 8.2 Save Processed Data for Next Steps
# Create the output directory if it doesn't exist
processed_dir = f'{BASE_PATH}/data/processed/'
os.makedirs(processed_dir, exist_ok=True)

# Use Parquet for DataFrames for speed and smaller file size
train_df.to_parquet(processed_dir + 'train_set.parquet', index=False)
test_df.to_parquet(processed_dir + 'test_set.parquet', index=False)

# Save the sparse matrix
scipy.sparse.save_npz(processed_dir + 'content_features.npz', content_features_sparse)

# Save the movieId to index mapping for later lookup
movie_id_map = pd.Series(aligned_genre_df.index, name='movieId')
movie_id_map.to_csv(processed_dir + 'movie_id_map.csv', index=True, header=True)

print(f"\nAll processed data has been saved to the '{processed_dir}' directory.")